In [ ]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2




r_seed = 42

WORKDIR = 'C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

In [ ]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [ ]:
## get dataset
pd.set_option("display.precision", 9)

# select only sensor data from smart watches
data = pd.read_csv(WORKDIR+'/Code/DataPrep/activity_recognition_data_watch_concat_by_comparacao_grupos.csv',dtype={"value": "float64"})

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
""" import random
def split_and_shuffle_3d_arrays(array_data, labels_data):
    random.seed(42)
    # Step 2: Combine the data and labels using zip
    combined = list(zip(array_data, labels_data))

    # Step 3: Shuffle the combined list
    random.shuffle(combined)

    # Step 4: Eliminate pairs with label change from 0 to 1 or 1 to 0
    filtered_combined = []
    for i in range(1, len(combined)):
        # Check if the labels in the current and previous windows change between 0 and 1 or vice versa
        previous_label = combined[i-1][1].flatten()
        current_label = combined[i][1].flatten()
        
        if not ((0 in previous_label and 2 in current_label) or (0 in previous_label and 3 in current_label) or
                (1 in previous_label and 2 in current_label) or (1 in previous_label and 3 in current_label) or
                (1 in previous_label and 0 in current_label) or (2 in previous_label and 0 in current_label) or
                (2 in previous_label and 1 in current_label) or (3 in previous_label and 1 in current_label) or
                (3 in previous_label and 2 in current_label)):
            filtered_combined.append(combined[i])

    # Step 5: Separate data and labels again and convert to numpy arrays with the same format
    filtered_data = np.array([x[0] for x in filtered_combined])
    filtered_labels = np.array([x[1] for x in filtered_combined])
    
    return filtered_data, filtered_labels

# This updated function now returns numpy arrays, preserving the original format.

 """

In [ ]:
def split_and_shuffle_3d_arrays(array_data, labels_data):
    
    combined = list(zip(array_data, labels_data))


    # Step 2: Separate into four parts based on label value
    # Filtering combined list by labels in each window
    grouped_data = {
        i: [item for item in combined if np.unique(item[1][:, 0])[0] == i] for i in range(4)
    }

    # Step 3: Interlace half of label=1 into half of label=0
    half_0 = len(grouped_data[0]) // 2
    half_1 = len(grouped_data[1]) // 2
    interlaced_0 = []

    for i in range(half_0):
        interlaced_0.append(grouped_data[0][i])
        if i < half_1:
            interlaced_0.append(grouped_data[1][i])

    # Step 4: Interlace half of label=3 into half of label=2
    half_2 = len(grouped_data[2]) // 2
    half_3 = len(grouped_data[3]) // 2
    interlaced_2 = []

    for i in range(half_2):
        interlaced_2.append(grouped_data[2][i])
        if i < half_3:
            interlaced_2.append(grouped_data[3][i])

    # Step 5: Interlace last half of label=0 into last half of label=3
    last_half_3 = grouped_data[3][half_3:]
    last_half_0 = grouped_data[0][half_0:]
    interlaced_3 = []

    for i in range(len(last_half_3)):
        interlaced_3.append(last_half_3[i])
        if i < len(last_half_0):
            interlaced_3.append(last_half_0[i])

    # Step 6: Concatenate the three resulting arrays vertically
    combined_result = interlaced_0 + interlaced_2 + interlaced_3

    # Step 7: Unzip the resulting array into separate data and labels
    data_result, labels_result = zip(*combined_result)

    # Converting back to numpy arrays for further processing if needed
    data_result = np.array(data_result)
    labels_result = np.array(labels_result)
    
    return data_result,labels_result



In [ ]:
import numpy as np

# Example array with binary class indications, replace this with your actual array
#array_data = np.random.randint(0, 2, (4027, 7)).astype(float)  # Example data; replace with your own

def count_active_classes(array_data):
    # Sum across each row to get the number of classes present in each row
    active_classes_per_row = np.sum(array_data, axis=1)
    
    # Count unique values in active_classes_per_row to get how many rows have each class count
    unique_counts, count_of_rows = np.unique(active_classes_per_row, return_counts=True)
    
    # Map each unique count of active classes to the number of rows with that count
    count_summary = dict(zip(unique_counts.astype(int), count_of_rows))
    
    return count_summary

# Get the count of active classes for each row
#active_classes_count = count_active_classes(array_data)

# Print the results
""" for active_class_count, row_count in active_classes_count.items():
    print(f"Number of rows with {active_class_count} active classes: {row_count}") """


In [ ]:
import numpy as np

def add_transition_windows(array_data, labels_data):
    # Lists to hold the modified data and labels
    modified_data = []
    modified_labels = []

    for i in range(len(array_data) - 1):
        # Append current window of data and labels
        modified_data.append(array_data[i])
        modified_labels.append(labels_data[i])

        # Check for label transition between consecutive windows
        if not np.array_equal(labels_data[i], labels_data[i + 1]):
            used_ratios = set()  # Track used ratios to avoid duplicates

            for _ in range(3):  # Repeat this process 4 times for different transitions
                # Generate a unique random split ratio
                while True:
                    ratio = np.random.rand()
                    if ratio not in used_ratios:
                        used_ratios.add(ratio)  # Add ratio to the set of used ratios
                        break  # Exit the loop once a unique ratio is found
                
                # Create empty transition windows with correct shapes
                transition_data = np.zeros((200, 6))
                transition_labels = np.zeros((200, 1))

                # Define split index using the unique ratio
                split_index = int(200 * ratio)

                # Fill transition window with previous activity data
                prev_data_sample = array_data[i]
                prev_label_sample = labels_data[i]
                transition_data[:split_index] = prev_data_sample[:split_index]
                transition_labels[:split_index] = prev_label_sample[:split_index]

                # Fill the rest with the next activity data
                next_data_sample = array_data[i + 1]
                next_label_sample = labels_data[i + 1]
                transition_data[split_index:] = next_data_sample[split_index:]
                transition_labels[split_index:] = next_label_sample[split_index:]

                # Append transition window to modified data and labels
                modified_data.append(transition_data)
                modified_labels.append(transition_labels)

                # Create mirrored versions of the transition data and labels
                mirrored_transition_data = np.flip(transition_data, axis=0)
                mirrored_transition_labels = np.flip(transition_labels, axis=0)

                # Append mirrored transition window to modified data and labels
                modified_data.append(mirrored_transition_data)
                modified_labels.append(mirrored_transition_labels)

    # Append the last window of data and labels
    modified_data.append(array_data[-1])
    modified_labels.append(labels_data[-1])

    # Convert lists to arrays
    final_data = np.array(modified_data)
    final_labels = np.array(modified_labels)

    return final_data, final_labels


In [ ]:
#print (shuffled_labels)

In [ ]:
#remove the null class
data=data.dropna()

# **Data Exploration**

In [ ]:
## the distribution of output classes
plt.bar(data["atividade"].value_counts().index, data["atividade"].value_counts().values)
plt.show()

In [ ]:
## x,y,z vaues for activity 'sit' 
data[(data['atividade'] == 0 )&(data['pessoa_x']==1600)][['acelerômetro_x','acelerômetro_Y','acelerômetro_Z']][:200].plot(title='0',subplots=True, figsize=(16, 12))

In [ ]:
## x,y,z vaues for activity 'stand' 
data[(data['atividade'] == 0 )&(data['pessoa_x']==1600)][['giroscópio_x','giroscópio_Y','giroscópio_Z']][:200].plot(title='0',subplots=True, figsize=(16, 12))

In [ ]:
data = data[data.pessoa_x!=1629]

In [ ]:
data.pessoa_x.unique()

In [ ]:
data = data.reset_index(drop=True)

In [ ]:
## number of events per user

plt.bar(data["pessoa_x"].value_counts().index, data["pessoa_x"].value_counts().values)
plt.show()

In [ ]:
#selecting only activities 0 to 3
data = data[data["atividade"].isin([0,2,3,4])]

In [ ]:
# Define the mapping dictionary
mapping_dict = {2:1,3:2,4:3}

In [ ]:
data["atividade"] = data["atividade"].replace(to_replace=mapping_dict)

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
grouped_data = data.groupby(['pessoa_x'])['atividade'].value_counts().unstack()

In [ ]:
print(grouped_data)

In [ ]:
nan_values = grouped_data[grouped_data.isnull().any(axis=1)].index

In [ ]:
nan_values_array = nan_values.to_numpy()

In [ ]:
print(nan_values)

In [ ]:
data = data[~data['pessoa_x'].isin([1637,1640,1642])]

In [ ]:
def prune_subjects_with_too_many_records(df):

    import pandas as pd

    # Assuming df is your DataFrame
    # Filtering for specific 'pessoa_x' values: 1621, 1628, and 1650
    filtered_df = df[(df['pessoa_x'].isin([1621, 1628, 1645,1650]))]

    # Keeping other 'pessoa_x' values between 1600 and 1650
    other_df = df[(df['pessoa_x'] >= 1600) & (df['pessoa_x'] <= 1650) & (~df['pessoa_x'].isin([1621, 1628, 1645,1650]))]

    # Group by 'pessoa_x' and 'atividade', and get first 3603 rows for each group
    filtered_limited_df = filtered_df.groupby(['pessoa_x', 'atividade']).head(3603)

    # Concatenate the filtered and the other dataframes
    result_df = pd.concat([filtered_limited_df, other_df])

    # Reset index if needed
    result_df = result_df.reset_index(drop=True)

    # Display the resulting DataFrame
    return result_df



In [ ]:
#data = prune_subjects_with_too_many_records(data)

In [ ]:
grouped_data2 = data.groupby(['pessoa_x'])['atividade'].value_counts().unstack()

In [ ]:
print(grouped_data2)

In [ ]:
def mark_mixed_windows(labels_data):
    mixed_count = 0
    count_2_3 = 0
    count_0_1 = 0
    count_3_0 = 0
    count_0_2 = 0
    count_0_3 = 0
    count_1_2 = 0
    count_1_3 = 0
    count_1_0 = 0
    count_2_0 = 0
    count_2_1 = 0
    count_3_1 = 0
    count_3_2 = 0
    
    # Loop over each window by index to modify the array directly
    for i in range(labels_data.shape[0]):
        window = labels_data[i]
        
        # Check if the first and last labels in the window match specific transitions
        first_label = window[0]  # First label in the window (since it's 1D now)
        last_label = window[-1]  # Last label in the window (1D array)
        
        # Check for transitions between specific label pairs
        if first_label == 2 and last_label == 3:
            labels_data[i] = 4  # Transition from 2 to 3
            count_2_3 += 1
        elif first_label == 0 and last_label == 1:
            labels_data[i] = 5  # Transition from 0 to 1
            count_0_1 += 1
        elif first_label == 3 and last_label == 0:
            labels_data[i] = 6  # Transition from 1 to 2
            count_3_0 += 1
        elif first_label == 0 and last_label == 2:
            labels_data[i] = 7  # Transition from 2 to 1
            count_0_2 += 1
        elif first_label == 0 and last_label == 3:
            labels_data[i] = 8
            count_0_3 += 1
        elif first_label == 1 and last_label == 2:
            labels_data[i] = 9
            count_1_2 += 1
        elif first_label == 1 and last_label == 3:
            labels_data[i] = 10  # Transition from 0 to 1
            count_1_3 += 1
        elif first_label == 1 and last_label == 0:
            labels_data[i] = 11  # Transition from 1 to 2
            count_1_0 += 1
        elif first_label == 2 and last_label == 0:
            labels_data[i] = 12  # Transition from 2 to 1
            count_2_0 += 1
        elif first_label == 2 and last_label == 1:
            labels_data[i] = 13
            count_2_1 += 1
        elif first_label == 3 and last_label == 1:
            labels_data[i] = 14
            count_3_1 += 1
        elif first_label == 3 and last_label == 2:
            labels_data[i] = 15
            count_3_2 += 1
            
    print(f"count_2_3 :{count_2_3}, count_0_1: {count_0_1}, count_3_0: {count_3_0}, count_0_2: {count_0_2} ")
    print(f"count_0_3 :{count_0_3}, count_1_2: {count_1_2}, count_1_3: {count_1_3}, count_1_0: {count_1_0} ")
    print(f"count_2_0 :{count_2_0}, count_2_1: {count_2_1}, count_3_1: {count_3_1}, count_3_2: {count_3_2} ")
          
    
    return mixed_count, labels_data

# This update fixes the issue by treating the `labels_data` as a 1D array for each window. The code should now work with your (453, 200) 2D array.


In [ ]:
import numpy as np

# Example array of shape (4027, 7) with categorical data in the second dimension
#array_data = np.random.randint(0, 7, (4027, 7))  # Replace with your actual data

def count_classes(array_data):
    # Flatten the array to count across all elements in the second dimension
    flattened_data = array_data.flatten()
    
    # Count unique values and their occurrences
    unique_classes, counts = np.unique(flattened_data, return_counts=True)
    
    # Create a dictionary to map classes to their counts
    class_counts = dict(zip(unique_classes, counts))
    
    return class_counts

# Get the counts for each class
#class_counts = count_classes(array_data)

# Print the counts



# **Models**

In [ ]:
import numpy as np

# Example labels data array of shape (3152, 200, 1)
#labels_data = np.random.randint(0, 3, (3152, 200, 1))  # Replace with your actual labels array

def count_mixed_windows(labels_data):
    mixed_count = 0
    
    # Loop over each window
    for window in labels_data:
        # Flatten the window to a 1D array and find unique label values
        unique_labels = np.unique(window)
        
        # Check if there is more than one unique label in the window
        if len(unique_labels) > 1:
            mixed_count += 1
    
    return mixed_count

# Count the mixed windows
#mixed_window_count = count_mixed_windows(labels_data)
#print(f"Number of mixed windows: {mixed_window_count}")


In [ ]:
def delete_segments_not_fully_filled(df_data_windowed,df_label_windowed):
    indexes = np.where(df_label_windowed == -1)[0]
    unique_arr = np.unique(indexes, return_index=True)[0]
    new_df_label_windowed = np.delete(df_label_windowed,unique_arr,axis=0)
    new_df_data_windowed = np.delete(df_data_windowed,unique_arr,axis=0)
    return (new_df_data_windowed,new_df_label_windowed)

In [ ]:
def apply_segmentation_no_sliding_window(df_data):
# Define window size and step size
    window_size = 200
    step_size = 20

    # Calculate number of windows
    n_windows = int((df_data.shape[0] / window_size))

    # Create segmented data using sliding window
    segments = np.zeros((n_windows, window_size, df_data.shape[1]-1),dtype=np.float32)
    targets = np.full((n_windows, window_size, 1),-1)
    for i in range(n_windows):
        start_idx = i * window_size
        end_idx = start_idx + window_size
        #populate the 3d array only if the 200 segment has the value in label
        label_consistency = (df_data[['atividade']][start_idx:end_idx].nunique()==1) 
        if label_consistency[0] == True:
            segments[i] = df_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']][start_idx:end_idx]
            targets[i] = df_data[['atividade']][start_idx:end_idx]
    return (segments,targets)

In [ ]:
def apply_segmentation_with_sliding_window(df_data):
    #creating sliding window for train data
    import numpy as np

    window_size = 200
    step_size = 20

    n_windows = int((df_data.shape[0] - window_size) // step_size + 1)

    segments = np.zeros((n_windows, window_size, df_data.shape[1]-1),dtype=np.float32)
    targets = np.full((n_windows, window_size, 1),-1)
    for i in range(n_windows):
        start_idx = i * step_size
        end_idx = start_idx + window_size
        label_consistency = df_data[['atividade']][start_idx:end_idx].nunique()==1
        if label_consistency[0] == True:
            segments[i] = df_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']][start_idx:end_idx]
            targets[i] = df_data[['atividade']][start_idx:end_idx]
    return (segments,targets)

In [ ]:
print(data.pessoa_x.unique())

In [ ]:
X_test_data = data[data['pessoa_x'].isin([1600,1601, 1602, 1603, 1604])]

In [ ]:
print(X_test_data.shape)

In [ ]:
X_test_data = X_test_data.reset_index(drop = True)

In [ ]:
X_test_data.head()

In [ ]:
X_test_data.tail()

In [ ]:
X_test_data.pessoa_x.unique()

In [ ]:
print(f"percentage of validation is {X_test_data.shape[0]/data.shape[0]}")

In [ ]:
X_test_data['atividade'].value_counts()

In [ ]:
X_test_data = X_test_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','atividade']]

In [ ]:
X_train_validation_data = data[~data.pessoa_x.isin([1600,1601, 1602, 1603, 1604])]

In [ ]:
print(X_train_validation_data.shape)

In [ ]:
X_train_validation_data = X_train_validation_data.reset_index(drop = True)

In [ ]:
X_train_validation_data.head()

In [ ]:
X_train_validation_data.tail()

In [ ]:
X_train_validation_data.pessoa_x.unique()

In [ ]:
print(f"percentage of validation is {X_train_validation_data.shape[0]/data.shape[0]}")

In [ ]:
X_train_validation_data['atividade'].value_counts()

In [ ]:
X_validation_data = X_train_validation_data[X_train_validation_data['pessoa_x'].isin([1605, 1606, 1607, 1608, 1609, 1610, 1611, 1612,1613])]

In [ ]:
print(X_validation_data.shape)

In [ ]:
X_validation_data = X_validation_data.reset_index(drop = True)

In [ ]:
X_validation_data.head()

In [ ]:
X_validation_data.tail()

In [ ]:
X_validation_data.pessoa_x.unique()

In [ ]:
print(f"percentage of validation is {X_validation_data.shape[0]/X_train_validation_data.shape[0]}")

In [ ]:
X_validation_data['atividade'].value_counts()

In [ ]:
X_validation_data = X_validation_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','atividade']]

In [ ]:
X_train_data = X_train_validation_data[~X_train_validation_data.pessoa_x.isin([1605, 1606, 1607, 1608, 1609, 1610, 1611, 1612,1613])]

In [ ]:
X_train_data.pessoa_x.unique()

In [ ]:
X_train_data.shape

In [ ]:
X_train_data = X_train_data.reset_index(drop=True)

In [ ]:
print(f"percentage of validation is {X_train_data.shape[0]/X_train_validation_data.shape[0]}")

In [ ]:
X_train_data.head()

In [ ]:
X_train_data.tail()

In [ ]:
X_train_data = X_train_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','atividade']]

In [ ]:
X_train_data.head()

In [ ]:
segments_test,targets_test = apply_segmentation_no_sliding_window(X_test_data)
print(targets_test.shape)
print(segments_test.shape)

In [ ]:
segments_test,targets_test = delete_segments_not_fully_filled(segments_test,targets_test)
print(targets_test.shape)
print(segments_test.shape)

In [ ]:
segments_test, targets_test = split_and_shuffle_3d_arrays(segments_test, targets_test)

In [ ]:
print(targets_test.shape)

In [ ]:
import numpy as np

def ocurrence_unique_values(targets_test):

    # Example of 3D array with the format (256, 200, 1)
    #data = np.random.randint(0, 3, (256, 200, 1))

    # Reshape to (256, 200) to simplify the processing
    data_reshaped = targets_test.reshape(targets_test.shape[0], 200)

    # Get unique values per window (each window has 200 rows)
    # Since the second dimension always contains the same value, we can just select the first column for each window
    unique_values = data_reshaped[:, 0]

    # Count occurrences of each unique value (0, 1, 2)
    unique, counts = np.unique(unique_values, return_counts=True)

    # Create a dictionary to show the counts of 0, 1, and 2
    occurrences = dict(zip(unique, counts))
    return occurrences


In [ ]:
ocurrences = ocurrence_unique_values(targets_test)
print(ocurrences)

In [ ]:
segments_test, targets_test = add_transition_windows(segments_test, targets_test)

In [ ]:
print(targets_test.shape)

In [ ]:
import numpy as np

def count_unique_value_windows(data):
    # Reshape to (n, 200) by flattening the third dimension (since it's (426, 200, 1))
    data_reshaped = data.reshape(data.shape[0], 200)
    
    # Initialize a dictionary to count occurrences of windows with unique values
    occurrences = {0.: 0, 1.: 0, 2.: 0, 3.: 0, 4.: 0, 5.: 0, 6.: 0, 7.: 0, 8.: 0, 9.: 0, 
                   10.: 0, 11.: 0, 12.: 0, 13.: 0, 14.: 0, 15.: 0}

    # Iterate through each window
    for window in data_reshaped:
        # Check if the window contains only one unique value
        unique_vals = np.unique(window)
        if len(unique_vals) == 1:  # If only one unique value in the window
            occurrences[unique_vals[0]] += 1  # Increment the count for that value
    
    return occurrences

# Now the function works with the input of shape (426, 200, 1). 



In [ ]:
count_unique_values = count_unique_value_windows(targets_test)

In [ ]:
print(count_unique_values)

In [ ]:
print(targets_test.shape)

In [ ]:
print(targets_test)

In [ ]:
mixed_window_count_teste, targets_test = mark_mixed_windows(targets_test)

In [ ]:
print(targets_test.shape)

In [ ]:
# Print the number of mixed windows
print(f"Number of mixed windows: {mixed_window_count_teste}")

# Example check of updated labels_data
print(targets_test)

In [ ]:
# Count transition windows
transition_window_count_teste = count_mixed_windows(targets_test)
print(f"Number of windows with transitions between labels: {transition_window_count_teste}")

In [ ]:
targets_test = targets_test.reshape(targets_test.shape[0],200)

In [ ]:
print(targets_test.shape)

In [ ]:
print(targets_test)

In [ ]:
labels_test = targets_test[:,0]

In [ ]:
print(labels_test)

In [ ]:
print(labels_test.shape)

In [ ]:
print(segments_test.shape)

In [ ]:
import numpy as np
def remove_useless_labels(labels_test,segments_test):

# Assuming array_3d is of shape (1239, 200, 6) and labels_1d is of shape (1239,)
    # Replace these with your actual arrays
    #array_3d = np.random.rand(1239, 200, 6)
    #labels_1d = np.random.randint(0, 9, size=(1239,))

    # Find the indices where labels are not 7 or 8
    valid_indices = np.where((labels_test != 7) & (labels_test != 8) & (labels_test != 9) 
                             & (labels_test != 10) & (labels_test != 11)
                             & (labels_test != 12) & (labels_test != 13)
                             & (labels_test != 14) & (labels_test != 15) )[0]

    # Filter both arrays
    filtered_labels_1d = labels_test[valid_indices]  # 1D array, indexed with 1 dimension
    filtered_array_3d = segments_test[valid_indices, :, :]  # 3D array, indexed with 3 dimensions

    return filtered_labels_1d, filtered_array_3d

    # Output the new shapes for verification
    #print("Filtered labels shape:", filtered_labels_1d.shape)
    #print("Filtered 3D array shape:", filtered_array_3d.shape)


In [ ]:
labels_test, segments_test = remove_useless_labels(labels_test,segments_test)

In [ ]:
print(segments_test.shape)

In [ ]:
print(labels_test.shape)

In [ ]:
print(segments_test.shape)

In [ ]:
import numpy as np

def reorganize_labels(labels_test):

    # Example of a 1D array with length 1239 and values from 0 to 6
    #array_1d = np.random.randint(0, 7, size=1239)

    # Replace values 0 to 2 with 0
    labels_test = np.where(labels_test <= 3, 0, labels_test)

    # Replace values 3 to 6 with 1, 2, 3, and 4 respectively
    mapping = {4: 1, 5: 2, 6: 3}
    labels_test = np.vectorize(mapping.get)(labels_test, labels_test)  # Only changes values 3 to 6

    return labels_test


In [ ]:
labels_test = reorganize_labels(labels_test)

In [ ]:
print(labels_test.shape)

In [ ]:
print(labels_test)

In [ ]:
labels_test_categorical = to_categorical(labels_test,num_classes=4)

In [ ]:
print(labels_test_categorical)

In [ ]:
print(labels_test_categorical.shape)

In [ ]:
import numpy as np

def category_count_categorical(labels_test_categorical):

    # Assuming your data is stored in a variable called `data`
    # Replace this with the actual data if it's available

    # Sum along the rows to get the count for each category
    category_counts_categorical = np.sum(labels_test_categorical, axis=0)

    # Print the counts
    print("Category counts:", category_counts_categorical)


In [ ]:
category_count_categorical(labels_test_categorical)

In [ ]:
segments_val,targets_val = apply_segmentation_no_sliding_window(X_validation_data)
print(targets_val.shape)
print(segments_val.shape)

In [ ]:
segments_val,targets_val = delete_segments_not_fully_filled(segments_val,targets_val)
print(targets_val.shape)
print(segments_val.shape)

In [ ]:
segments_val, targets_val = split_and_shuffle_3d_arrays(segments_val, targets_val)

In [ ]:
ocurrences = ocurrence_unique_values(targets_test)
print(ocurrences)

In [ ]:
segments_val, targets_val = add_transition_windows(segments_val, targets_val)       

In [ ]:
print(targets_test.shape)

In [ ]:
count_unique_values = count_unique_value_windows(targets_val)

In [ ]:
print(count_unique_values)


In [ ]:
print(targets_test.shape)

In [ ]:
print(targets_test)

In [ ]:
mixed_window_count_val, targets_val = mark_mixed_windows(targets_val)

In [ ]:
print(targets_val.shape)

In [ ]:
# Print the number of mixed windows
print(f"Number of mixed windows: {mixed_window_count_val}")

# Example check of updated labels_data
print(targets_val)

In [ ]:
# Example check of updated labels_data
print(targets_val)


In [ ]:
# Count transition windows
transition_window_count_val = count_mixed_windows(targets_val)
print(f"Number of windows with transitions between labels: {transition_window_count_val}")

In [ ]:
targets_val = targets_val.reshape(targets_val.shape[0],200)

In [ ]:
print(targets_val.shape)

In [ ]:
print(targets_val)

In [ ]:
labels_val = targets_val[:,0]

In [ ]:
print(labels_val)

In [ ]:
print(labels_val.shape)

In [ ]:
labels_val, segments_val = remove_useless_labels(labels_val,segments_val)

In [ ]:
labels_val = reorganize_labels(labels_val)

In [ ]:
labels_val_categorical = to_categorical(labels_val,num_classes=4)

In [ ]:
print(labels_val_categorical)

In [ ]:
print(labels_val_categorical.shape)

In [ ]:
category_count_categorical(labels_val_categorical)


In [ ]:
segments_train,targets_train = apply_segmentation_no_sliding_window(X_train_data)

In [ ]:
print(segments_train.shape)
print(targets_train.shape)

In [ ]:
segments_train, targets_train = delete_segments_not_fully_filled(segments_train,targets_train)

In [ ]:
segments_train, targets_train  = split_and_shuffle_3d_arrays(segments_train , targets_train )


In [ ]:

ocurrences = ocurrence_unique_values(targets_train )
print(ocurrences)

In [ ]:
print(segments_train.shape)
print(targets_train.shape)

In [ ]:
segments_train , targets_train  = add_transition_windows(segments_train , targets_train )

In [ ]:

print(targets_train.shape)

In [ ]:
count_unique_values = count_unique_value_windows(targets_train )

In [ ]:
print(count_unique_values)

In [ ]:
print(targets_train.shape)



In [ ]:
print(targets_train )


In [ ]:
mixed_window_count_teste, targets_train  = mark_mixed_windows(targets_train )

In [ ]:
print(targets_train.shape)

In [ ]:
# Print the number of mixed windows
print(f"Number of mixed windows: {mixed_window_count_teste}")

In [ ]:
# Example check of updated labels_data
print(targets_train )

In [ ]:
# Count transition windows
transition_window_count_teste = count_mixed_windows(targets_train )
print(f"Number of windows with transitions between labels: {transition_window_count_teste}")

In [ ]:
#targets_train_teste1 = targets_train.reshape(targets_train.shape[0],200)

In [ ]:
#print(targets_train_teste1.shape)

In [ ]:
#labels_train_teste1 = targets_train_teste1[:,0]

In [ ]:
#labels_train_categorical_teste1 = to_categorical(labels_train_teste1,num_classes=3)

In [ ]:
#print(labels_train_categorical_teste1.shape)

In [ ]:
#print(labels_train_categorical_teste1)

In [ ]:
targets_train = targets_train.reshape(targets_train.shape[0],200)

In [ ]:
print(targets_train.shape)

In [ ]:
labels_train = targets_train[:,0]

In [ ]:
labels_train , segments_train  = remove_useless_labels(labels_train ,segments_train )

In [ ]:
labels_train  = reorganize_labels(labels_train )

In [ ]:
labels_train_categorical = to_categorical(labels_train,num_classes=4)

In [ ]:
print(labels_test_categorical)

In [ ]:
categ_count = category_count_categorical(labels_train_categorical)
print(categ_count)


In [ ]:
print(labels_train_categorical.shape)

In [ ]:
print(labels_train_categorical)

In [ ]:
X_train = segments_train
y_train = labels_train_categorical
X_validation = segments_val
y_validation = labels_val_categorical

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

In [ ]:
def shuffle_data(variable_data,label_data):
    perm_indices = np.random.permutation(variable_data.shape[0])
    # Shuffle both arrays using the same permutation indices
    shuffled_variable_data = variable_data[perm_indices]
    shuffled_label_data = label_data[perm_indices]
    return (shuffled_variable_data,shuffled_label_data)

In [ ]:
X_train,y_train = shuffle_data(X_train,y_train)
X_validation,y_validation = shuffle_data(X_validation,y_validation)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

In [ ]:
print(y_train.shape)

In [ ]:
print(X_train.shape)

In [ ]:
#X_validation1, y_validation1 = add_transition_windows(X_validation, y_validation)

In [ ]:
#print(shuffled_labels)

In [ ]:
#from tensorflow.keras import regularizers

In [ ]:
''' def cria_LSTM(hp):
  model = tf.keras.Sequential()  
  model.add(tf.keras.layers.LSTM(hp.Int("input_units",min_value=32,max_value=256,step=32), input_shape=(200,6), return_sequences=False))
  model.add(tf.keras.layers.Dense(hp.Int("input_units_relu",min_value=32,max_value=256,step=32), activation='relu'))
  model.add(tf.keras.layers.BatchNormalization())  
  model.add(tf.keras.layers.Dense(units=4, activation='softmax'))
  model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
  return model
 '''

In [ ]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Activation, Permute, Multiply, Lambda, Reshape
import keras.backend as K

def lstm_attention_sensor(hp):
    inputs = Input(shape=(200,6))
    lstm_out = LSTM(hp.Int("input_units_lstm1",min_value=32,max_value=512,step=32), kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
             recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),return_sequences=True)(inputs)
    
    # Attention mechanism
    attention = Dense(hp.Int("input_units_lstm2",min_value=32,max_value=512,step=32), activation='relu')(lstm_out)    
    attention = Permute([2, 1])(attention)
    attention = Activation('softmax')(attention)
    attention = Permute([2, 1])(attention)
    
    context_vector = Multiply()([lstm_out, attention])
    context_vector = Lambda(lambda x: K.sum(x, axis=1))(context_vector)    
    output = Dense(4, activation='softmax')(context_vector)
    
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
    return model


In [ ]:
tuner = RandomSearch(
    lstm_attention_sensor,
    objective = 'val_accuracy', 
    seed=42,
    max_trials = 7,
    overwrite=True,        
    executions_per_trial = 1,    
    directory = LOG_DIR,
    project_name="lstm_acel_giro_multiclass")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    min_delta = 0,
    verbose=1,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    verbose=1)

tensorboard = tf.keras.callbacks.TensorBoard(LOG_DIR)

my_callbacks = [early_stopping,reduce_lr,tensorboard]  


In [ ]:
tuner.search(x=X_train,
            y=y_train,
            epochs=50,
            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=32, use_multiprocessing=True)

In [ ]:
tuner.results_summary()

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.history

In [ ]:
best_model.summary()

In [ ]:
loss, accuracy = best_model.evaluate(X_validation, y_validation)

In [ ]:
predictions = best_model.predict(X_validation)

In [ ]:
print(y_validation)

In [ ]:
print(y_validation.shape)

In [ ]:
y_validation_inverse = np.argmax(y_validation, axis=1)

In [ ]:
print(y_validation_inverse)

In [ ]:
print(len(y_validation_inverse))

In [ ]:
print(predictions)

In [ ]:
classes_x=np.argmax(predictions,axis=1)

In [ ]:
print(classes_x)

In [ ]:
print(len(classes_x))

In [ ]:
print(classes_x.shape)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm = confusion_matrix(y_validation_inverse, classes_x)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

In [ ]:
disp.plot()

In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
print(segments_test)

In [ ]:
print(labels_test_categorical.shape)

In [ ]:
print(labels_test_categorical)

In [ ]:
loss_test, accuracy_test = best_model.evaluate(segments_test, labels_test_categorical)

In [ ]:
predictions_test = best_model.predict(segments_test)

In [ ]:
print(predictions_test)

In [ ]:
classes_x_test=np.argmax(predictions_test,axis=1)

In [ ]:
print(classes_x_test)

In [ ]:
labels_test_argmax = np.argmax(labels_test_categorical,axis=1)

In [ ]:
print(labels_test_argmax)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm_test = confusion_matrix(labels_test_argmax, classes_x_test)

In [ ]:
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test)
disp_test.plot()

In [ ]:
print(classification_report(labels_test_argmax, classes_x_test))

In [ ]:
best_model.save(WORKDIR+"/Model/best_models_saved_"+data_hora)